# Colab starter

## 1. Confirm the GPU
Free-tier Colab assigns a T4-class GPU (availability varies by time of day).

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 2. Mount Drive (persistent storage)
Everything expensive (datasets, checkpoints) goes here, not in `/content`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/dip-p28'
DATA = os.path.join(ROOT, 'data')
CKPT = os.path.join(ROOT, 'checkpoints')
os.makedirs(DATA, exist_ok=True)
os.makedirs(CKPT, exist_ok=True)
print('DATA =', DATA)
print('CKPT =', CKPT)

## 3. Clone the team repo
The repo is public, but the notebook still uses a fine-grained GitHub token for the clone and for pushing results (cell 8). Create one at github.com/settings/tokens with **Contents: Read and write** for `mo1998/New-No-Reference-Image-Quality-Metric` and store it as the Colab secret `GH_TOKEN` (key icon -> Add new secret -> enable notebook access). **Never print or commit the token.** If the secret is unavailable (e.g. in VS Code), the cell falls back to a token previously saved in your Drive (`dip-p28/.gh_token`) and, as a last resort, to a masked prompt - which then saves it to your Drive, so you only ever type it once.

In [ ]:
from google.colab import userdata
from getpass import getpass
import os, subprocess

TOKEN_FILE = '/content/drive/MyDrive/dip-p28/.gh_token'  # private copy in your Drive

def load_token():
    # 1) Colab secret (preferred when running from the Colab UI)
    try:
        return userdata.get('GH_TOKEN')
    except Exception as e:
        print('Colab secret unavailable:', type(e).__name__)
    # 2) token saved in your Drive from a previous session - no typing needed
    if os.path.exists(TOKEN_FILE):
        print('Using token saved in Drive:', TOKEN_FILE)
        return open(TOKEN_FILE).read().strip()
    # 3) ask once, then save so you never type it again
    token = getpass('Paste GH_TOKEN (hidden; it will be saved in your Drive): ')
    try:
        with open(TOKEN_FILE, 'w') as f:
            f.write(token)
        print('Token saved for future sessions:', TOKEN_FILE)
    except OSError as e:
        print('Could not save token to Drive:', e)
    return token

token = load_token()
repo = 'mo1998/New-No-Reference-Image-Quality-Metric'
clone_url = f'https://{token}@github.com/{repo}.git'   # token-bearing, for the clone only
clean_url = f'https://github.com/{repo}.git'           # stored as origin (no token)

# Work from a valid directory first: a previous session may have deleted the cwd
os.chdir('/content')
subprocess.run(['rm', '-rf', '/content/repo'], check=True)
r = subprocess.run(['git', 'clone', '--quiet', clone_url, '/content/repo'],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise RuntimeError('git clone failed: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', '/content/repo', 'remote', 'set-url', 'origin', clean_url], check=True)
os.chdir('/content/repo')
print(subprocess.run(['git', '-C', '/content/repo', 'log', '--oneline', '-3'],
                     capture_output=True, text=True).stdout)

## 4. Install project dependencies
Colab ships PyTorch already; we add the IQA metric packages from `requirements.txt`.

In [ ]:
!pip install --quiet lpips pyiqa
import lpips, pyiqa
print('lpips and pyiqa ready')

## 5. Fixed seed

In [ ]:
import sys
sys.path.insert(0, '/content/repo/src')
from utils import set_seed, SEED
set_seed(SEED)
print('seed set:', SEED)

## 6. KADID-10k dataset
Official source: [datasets.vqa.mmsp-kn.de/archives/kadid10k.zip](https://datasets.vqa.mmsp-kn.de/archives/kadid10k.zip) (~3.1 GB). Citation: Lin, Hosu & Saupe, *KADID-10k: A Large-scale Artificially Distorted IQA Database*, QoMEX 2019. Freely available to the research community - cite it in the report.
NOTE (2026-10-10): the authors' host serves an expired TLS certificate, so wget must run with `--no-check-certificate`; we compensate by verifying the content - (a) exact byte size, (b) zip CRC test, (c) SHA-256 of `dmos.csv` against the authors' OSF mirror (valid TLS): https://osf.io/xkqjh/ (linked as 'Alternative download' on the official page).
Real archive layout (verified 2026-10-10): one top-level `kadid10k/` folder containing `images/` - 81 reference PNGs (`Ixx.png`), 10,125 distorted PNGs (`Ixx_yy_zz.png`), plus ~132 macOS `._*` metadata files (harmless, ignored) - and `dmos.csv` with columns `dist_img, ref_img, dmos, var` (dmos = human quality label, range 1-5, higher = better).
The zip is stored once in Drive; each session copies it to `/content` and unzips locally (fast reads; 10k small files are slow to stream from Drive every epoch).

In [ ]:
import os, glob, shutil, subprocess, hashlib, re

URL = 'https://datasets.vqa.mmsp-kn.de/archives/kadid10k.zip'
ZIP_LOCAL = '/content/kadid10k.zip'
ZIP_DRIVE = os.path.join(DATA, 'kadid10k.zip')  # DATA defined in cell 2
EXPECTED_BYTES = 3067408471                    # exact size served by the host
DMOS_SHA256 = '573e2ed98fdaa2a5aed7b50ad716906e125e1001ed0e97c8e93e79d7b997ba99'  # authors' OSF copy

# 1) Get the zip: reuse the Drive copy if we already downloaded it once
if os.path.exists(ZIP_DRIVE):
    print('Zip already in Drive - copying to local disk...')
    subprocess.run(['cp', ZIP_DRIVE, ZIP_LOCAL], check=True)
else:
    print('Downloading KADID-10k (~3.1 GB)...')
    # The authors' host has an EXPIRED TLS certificate, so wget refuses it.
    # Bypass cert verification, then verify the CONTENT:
    #   (a) exact byte size served by the host, (b) zip CRC test,
    #   (c) sha256 of dmos.csv vs the authors' OSF mirror (valid TLS).
    subprocess.run(['wget', '-c', '--no-check-certificate', URL, '-O', ZIP_LOCAL], check=True)
    size = os.path.getsize(ZIP_LOCAL)
    assert size == EXPECTED_BYTES, f'size mismatch: {size} != {EXPECTED_BYTES}'
    print('Size OK:', size, 'bytes')
    print('CRC test...')
    subprocess.run(['unzip', '-tq', ZIP_LOCAL], check=True)
    print('Copying zip to Drive (persistent storage)...')
    subprocess.run(['cp', ZIP_LOCAL, ZIP_DRIVE], check=True)

# 2) Unzip to local disk for this session. The archive nests everything under
#    one top-level folder -> locate it via dmos.csv instead of hard-coding.
shutil.rmtree('/content/kadid10k', ignore_errors=True)
subprocess.run(['unzip', '-q', ZIP_LOCAL, '-d', '/content/kadid10k'], check=True)
csv_hits = glob.glob('/content/kadid10k/*/dmos.csv')
assert len(csv_hits) == 1, f'unexpected archive layout: {csv_hits}'
CSV_PATH = csv_hits[0]
IMG_DIR = os.path.join(os.path.dirname(CSV_PATH), 'images')

# 3) Verify contents
pngs  = [f for f in os.listdir(IMG_DIR) if f.lower().endswith('.png')]
refs  = [f for f in pngs if re.fullmatch(r'i\d{2}\.png', f, re.I)]
dists = [f for f in pngs if re.fullmatch(r'i\d{2}_\d{2}_\d{2}\.png', f, re.I)]
mac   = [f for f in pngs if f.startswith('._')]
rows  = sum(1 for _ in open(CSV_PATH)) - 1
h = hashlib.sha256(open(CSV_PATH, 'rb').read()).hexdigest()
print('refs:', len(refs), '(expected 81)')
print('distorted:', len(dists), '(expected 10,125)')
print('macOS metadata files ignored:', len(mac))
print('DMOS rows:', rows, '(expected 10,125)')
print('dmos.csv sha256 matches authors OSF copy:', h == DMOS_SHA256)
print('CSV_PATH =', CSV_PATH)
print('IMG_DIR  =', IMG_DIR)

## 7. M1 baselines - PSNR / SSIM / LPIPS vs DMOS
Runs `src/run_m1_baselines.py`: content split by reference image (seed 42), computes the three full-reference baselines on the test split, and reports SROCC / PLCC / RMSE against the human DMOS labels. Add `--full` to also evaluate all 10,125 images (slower).

In [ ]:
!cd /content/repo && python src/run_m1_baselines.py --data-root /content/kadid10k/kadid10k --out results/m1_baseline_correlations.csv

## 8. Push results to the repo (needs a Contents: Read-and-write token)
The token from cell 3 is handed to git out-of-band via GIT_ASKPASS, so it never appears in the command line, the repo config, or the output.

In [ ]:
import os, subprocess

def git(*args):
    return subprocess.run(['git', '-C', '/content/repo'] + list(args),
                          capture_output=True, text=True).stdout.strip()

changes = git('status', '--porcelain', 'results')
unpushed = git('log', '--oneline', 'origin/main..HEAD')
print('changes in results/:')
print(changes or '(none)')
if unpushed:
    print('unpushed commits:')
    print(unpushed)

if not changes and not unpushed:
    print('Nothing to commit or push. Run the M1 baselines cell (## 7) first, then re-run this cell.')
else:
    # token handed to git via GIT_ASKPASS, never on the command line
    open('/content/askpass.sh', 'w').write('#!/bin/sh\necho "$GH_PUSH_TOKEN"\n')
    os.chmod('/content/askpass.sh', 0o700)
    env = dict(os.environ, GIT_ASKPASS='/content/askpass.sh', GH_PUSH_TOKEN=token)
    # guarantee the origin URL is correct (older sessions stored a malformed one)
    want = 'https://github.com/mo1998/New-No-Reference-Image-Quality-Metric.git'
    if git('remote', 'get-url', 'origin') != want:
        print('fixing origin URL')
        subprocess.run(['git', '-C', '/content/repo', 'remote', 'set-url', 'origin', want], check=True)
    if changes:
        subprocess.run(['git', '-C', '/content/repo', 'config', 'user.name', 'mo1998'], env=env, check=True)
        subprocess.run(['git', '-C', '/content/repo', 'config', 'user.email', 'mo1998@users.noreply.github.com'], env=env, check=True)
        subprocess.run(['git', '-C', '/content/repo', 'add', 'results'], env=env, check=True)
        subprocess.run(['git', '-C', '/content/repo', 'commit', '-m', 'results: M1 baseline correlations (PSNR/SSIM/LPIPS vs DMOS, content split, seed 42)'], env=env, check=True)
    # integrate commits pushed elsewhere since our clone (fast-forward or rebase)
    pr = subprocess.run(['git', '-C', '/content/repo', 'pull', '--rebase', 'origin', 'main'],
                        env=env, capture_output=True, text=True)
    print('pull --rebase exit:', pr.returncode)
    if pr.returncode != 0:
        print(pr.stderr[-800:])
    p = subprocess.run(['git', '-C', '/content/repo', 'push', 'origin', 'main'],
                       env=env, capture_output=True, text=True)
    print('push exit:', p.returncode)
    if p.returncode == 0:
        print(git('log', '--oneline', '-1'))
        print('results pushed to GitHub')
    else:
        print('--- push stderr (last 1500 chars) ---')
        print(p.stderr[-1500:])